# Creating Spark Session

In [0]:
%pip install beautifulsoup4 textacy nltk transformers SentencePiece tqdm torch spacy

import spacy
spacy.cli.download("en_core_web_sm")
dbutils.library.restartPython()

In [0]:
import os
os.environ['TRANSFORMERS_CACHE'] = '/dbfs/Volumes/workspace/default/ensf612'
os.environ['HF_HOME'] = '/dbfs/Volumes/workspace/default/ensf612'

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.config("spark.sql.session.timeZone", "UTC").config("spark.sql.execution.arrow.maxRecordsPerBatch", 8).appName("FinSentAnalysis").getOrCreate()

# Get Data

### Get News Data

In [0]:
# global constants
API_KEY : str = ''
BEZINGA_URL : str = 'https://api.benzinga.com/api/v2/news'
STOCK_TICKER : str = 'AAPL'
HEADERS = {"accept": "application/json"}
PAGE_SIZE = 100
PAGE_LIMIT = 400

params = {
    'token': API_KEY,
    'displayOutput' : 'full',
    'pageSize' : PAGE_SIZE,
    "sort": "created:asc",
    'tickers': STOCK_TICKER,
    'channels' : 'news'
}

In [0]:
import requests

news_data = []

params['dateFrom'] = '2015-01-01'
params['dateTo'] = '2020-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()

params['dateFrom'] = '2021-01-01'
params['dateTo'] = '2023-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()

params['dateFrom'] = '2024-01-01'
params['dateTo'] = '2025-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()


In [0]:
import json

# Define the filename
dbfs_path = "dbfs:/Volumes/workspace/default/ensf612/aapl_news.json"

dbutils.fs.put(dbfs_path, json.dumps(news_data), overwrite=True)

### Get Stock Data

In [0]:
# global constants
API_KEY : str = ''
API_SECRET_KEY : str = ''
STOCK_TICKER : str = 'AAPL'
ALPACA_URL : str = f'https://data.alpaca.markets/v2/stocks/{STOCK_TICKER}/bars'
HEADERS = {
    "accept": "application/json",
    "APCA-API-KEY-ID": API_KEY,
    "APCA-API-SECRET-KEY": API_SECRET_KEY
    }

params = {
    'timeframe' : '1D',
    'limit' : 10000,
    'adjustment' : 'all'
}

In [0]:
import requests

stock_data = []

params['start'] = '2015-01-01'
params['end'] = '2018-12-31'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock1 = response.json()['bars']
stock_data += stock1

params['start'] = '2019-01-01'
params['end'] = '2021-12-31'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock2 = response.json()['bars']
stock_data += stock2

params['start'] = '2022-01-01'
params['end'] = '2025-11-05'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock3 = response.json()['bars']
stock_data += stock3

In [0]:
import json

# Define the filename
dbfs_path = "dbfs:/Volumes/workspace/default/ensf612/aapl_price.json"

dbutils.fs.put(dbfs_path, json.dumps(stock_data), overwrite=True)

---

# Cataloging Data

In [0]:
import pandas as pd

workspace = 'danish.shahid@ucalgary.ca'

news_df = pd.read_json(f"../data/aapl_news.json")
price_df = pd.read_json(f"../data/aapl_price.json")

sp_news_df = spark.createDataFrame(news_df)
sp_price_df = spark.createDataFrame(price_df)

sp_news_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_news.json", mode="overwrite")
sp_price_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_price.json", mode="overwrite")

# Reading Data

In [0]:
news_df = spark.read.json("dbfs:/Volumes/workspace/default/ensf612/aapl_news.json").select(*['id', 'created', 'title', 'teaser', 'body'])
price_df = spark.read.json("dbfs:/Volumes/workspace/default/ensf612/aapl_price.json")

### Fixing time stamps

In [0]:
news_df = news_df.withColumn('created', regexp_replace('created', r"^[A-Za-z]{3},\s+", "")).withColumn('created', to_timestamp('created', "dd MMM yyyy HH:mm:ss Z")).orderBy('created')
price_df = price_df.withColumn('t', to_timestamp('t', "yyyy-MM-ddTHH:mm:ssZ")).orderBy('t')

### Removing HTML tagging

In [0]:
from bs4 import BeautifulSoup as bs

@udf
def parseHTML(text):
  return bs(text, 'html.parser').get_text()


news_df = news_df.withColumn('body', parseHTML(col('body'))).withColumn('teaser', parseHTML(col('teaser'))).withColumn('title', parseHTML(col('title')))

### Removing New Line Characters

In [0]:
news_df = news_df.withColumn('body', regexp_replace('body', r'\r|\n|\t', ' ')).withColumn('teaser', regexp_replace('teaser', r'\r|\n|\t', ' ')).withColumn('title', regexp_replace('title', r'\r|\n|\t', ' '))

### Removing urls

In [0]:
from textacy.preprocessing import *

@udf
def removeUrls(text):
  return replace.urls(text)

news_df = news_df.withColumn('body', removeUrls('body')).withColumn('teaser', removeUrls('teaser')).withColumn('title', removeUrls('title'))

### Removing brackets

In [0]:
@udf
def removeBrackets(text):
  return remove.brackets(text)

news_df = news_df.withColumn('body', removeBrackets('body')).withColumn('teaser', removeBrackets('teaser')).withColumn('title', removeBrackets('title'))

In [0]:
@udf
def normalizeWhitespaces(text):
  return normalize.whitespace(text)

news_df = news_df.withColumn('body', normalizeWhitespaces('body')).withColumn('teaser', normalizeWhitespaces('teaser')).withColumn('title', normalizeWhitespaces('title'))

In [0]:
@udf
def normalizeHyphen(text):
  return normalize.hyphenated_words(text)

news_df = news_df.withColumn('body', normalizeHyphen('body')).withColumn('teaser', normalizeHyphen('teaser')).withColumn('title', normalizeHyphen('title'))

In [0]:
import re

@udf
def replaceImageText(text):
    return re.sub(r'Image:.*|Also Read: ', '', text)

news_df = news_df.withColumn('body', replaceImageText('body'))

In [0]:
import en_core_web_sm

nlp = en_core_web_sm.load()

APPLE_NAMES = {
    "apple",
    "apple inc.",
    "apple, inc.",
    "apple incorporated",
}

def is_aapl_sentence(span):
    """
    Decide if a sentence is about Apple stock / company.
    Heuristics:
      - contains ticker 'AAPL'
      - or has ORG/PRODUCT entity with Apple name
    """
    text_lower = span.text.lower()

    # Check explicit ticker mention
    if "aapl" in text_lower or "apple" in text_lower:
        return True

    # Check NER entities
    for ent in span.ents:
        if ent.label_ in ("ORG", "PRODUCT"):
            if ent.text.lower() in APPLE_NAMES:
                return True

    return False

@udf
def split_sentences(text):
    doc = nlp(text)
    return '|'.join([sent.text.strip() for sent in doc.sents if is_aapl_sentence(sent)])


news_df = news_df.withColumn('body', split(split_sentences('body'), r'\|'))

In [0]:
news_df.filter(expr("created > '2025-06-01'")).limit(10).display()

In [0]:
from transformers import pipeline
import torch
device = 0 if torch.cuda.is_available() else -1

_classifier = None

def get_classifier():
    """
    Lazily initialize the FinBERT pipeline on each worker.
    This function runs ON THE EXECUTOR, not the client,
    so the model is not serialized over gRPC.
    """
    global _classifier
    if _classifier is None:
        _classifier = pipeline("text-classification", model="ProsusAI/finbert", top_k=1, device=device)
    return _classifier

In [0]:
import pandas as pd
#from tqdm.auto import tqdm

@pandas_udf(ArrayType(MapType(StringType(), DoubleType())))
def classify_text_udf(bodies: pd.Series) -> pd.Series:
    """
    bodies: each row is a Python list of sentences (Spark array<string>)

    returns: each row is a list of maps:
      [
        {"positive": 0.1, "negative": 0.8, "neutral": 0.1},
        {"positive": 0.6, "negative": 0.2, "neutral": 0.2},
        ...
      ]
    """
    out = []
    clf = get_classifier()  # model created on worker the first time

    for sentences in bodies:
        # handle nulls
        if sentences is None:
            out.append(None)
            continue

        if isinstance(sentences, str):
            sentences = [sentences]

        # run FinBERT on the list of sentences
        preds = clf(
            sentences
        )

        sentence_maps = []
        for per_sentence in preds:
            # per_sentence is a list like:
            # [{"label": "positive", "score": ...}, {"label": "negative", ...}, ...]
            label_scores = {item["label"]: float(item["score"]) for item in per_sentence}
            sentence_maps.append(label_scores)

        out.append(sentence_maps)

    return pd.Series(out)

In [0]:
news_df_part = news_df#.repartition(int((news_df.count()/20) + 1))
#news_df_part.count()
news_df_trunc = news_df_part.withColumn('sentiment_body', classify_text_udf('body')).withColumn('sentiment_title', classify_text_udf('title')).withColumn('sentiment_teaser', classify_text_udf('teaser'))

In [0]:
news_df_trunc.display()

In [0]:
news_df_trunc.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("aapl_sentiment")

In [0]:

import pandas as pd
from pyspark.sql.functions import pandas_udf
from tqdm.auto import tqdm
 
@pandas_udf('string')
def classify_batch_udf(texts: pd.Series) -> pd.Series:
  pipe = classifier(texts.to_list())
  summaries = [{'label': pred['label'], 'score' : pred['score']} for pred in pipe]
  return pd.Series(summaries)

In [0]:
def classify_batch_udf(text):
  pipe = classifier(text)
  summaries = [{'label': pred['label'], 'score' : pred['score']} for pred in pipe]
  return summaries[0]

In [0]:
def truncate_text(text, max_length=512):
    # Truncate text to the first max_length characters or tokens
    return text[:max_length]

In [0]:
news_pd = news_df.filter(expr("created > '2025-06-01'")).toPandas()
news_pd["sentiment_body"] = news_pd["body"].apply(lambda x: [classify_batch_udf(y) for y in x])

In [0]:
news_pd["sentiment_title"] = news_pd["title"].apply(lambda x: classify_batch_udf(x))
news_pd["sentiment_teaser"] = news_pd["teaser"].apply(lambda x: classify_batch_udf(x))

In [0]:
display(news_pd)

In [0]:
news_pd['sentiment_body']

In [0]:
display(news_pd)

In [0]:
news_df_trunc = news_df.filter(expr("created > '2025-06-01'")).limit(1).withColumn('sentiment_body', classify_body_udf('body'))

In [0]:
display(news_df_trunc.toPandas())